In [1]:
import math
import os
import shutil

import gemmi
import matplotlib.pyplot as plt
import mrcfile
import numpy as np

In [2]:
map_path = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-11893_re_3.33/emd_11893_normalized.mrc'
model_path = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-11893_re_3.33/7ase.cif'
model_parts = ['unit_1', 'unit_2']
sample_dir = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/ready_to_train_and_val'
npy_size = 64

In [3]:
with mrcfile.mmap(map_path) as mrc:
    map_data = mrc.data
    # head_nz, head_ny, head_nx = mrc.header.nz, mrc.header.ny, mrc.header.nx

    # Ensure the MRC file has a cube shape
    if not (mrc.header.nz == mrc.header.ny == mrc.header.nx):
        raise ValueError("MRC file is not cubic.")
    cube_size = [int(mrc.header.nz), int(mrc.header.ny), int(mrc.header.nx)]

In [5]:
structure = gemmi.read_structure(model_path)

In [6]:
# Ensure the size of the model is not larger than the map
box = structure.calculate_box()
box_min = list(box.minimum)[::-1]
box_max = list(box.maximum)[::-1]
if box_max[2] > cube_size[2] or box_max[1] > cube_size[2] or box_max[0] > cube_size[0]:
    raise ValueError("The model box size exceeds that of the map.")


In [7]:
def compute_grid_params(box_min_list, box_max_list, axis_length_list, grid_size):
    """
    Computes the starting coordinate and number of grid samples along three axis for a given box.

    Args:
        box_min_list (List): Minimum coordinates of the box.
        box_max_list (List): Maximum coordinates of the box.
        axis_length_list (List): Lengths of the axes.
        grid_size (int): Size of the grid.

    Returns:
        Tuple[List[int], List[int]]: Starting coordinates and numbers of grid samples when sampling stride = 0.
    """
    start_coords = []
    n_samples = []
    for box_min, box_max, axis_length in zip(box_min_list, box_max_list, axis_length_list):
        box_mid = (box_min + box_max) // 2
        n_samples_axis = (box_max - box_min) // grid_size + 1
        if not (32 * n_samples_axis <= box_mid <
                axis_length - 32 * n_samples_axis):
            n_samples_axis -= 1
        start_coord = box_mid - 32 * n_samples_axis
        start_coords.append(int(start_coord))
        n_samples.append(int(n_samples_axis))
    return start_coords, n_samples

In [8]:
# Compute the grid parameters for splitting the volume
start_coords, n_samples = compute_grid_params(box_min, box_max, cube_size, npy_size)
print(start_coords, n_samples)

[103, 51, 77] [4, 5, 5]


In [10]:
def split_to_npy(data,
                 sample_dir,
                 start_coords,
                 n_samples,
                 npy_size,
                 sample_num,
                 part_name,
                 extract_stride=32):
    """
    Extracts sub-volumes of size npy_size from the input data array, starting from the given start coordinates and generates npy files for each sub-volume.

    Parameters:
        data (ndarray): Input data array
        sample_dir (str): Directory to save the npy files
        start_coords (tuple): Tuple of start coordinates (z,y,x)
        n_samples (tuple): Tuple of number of samples to extract (z,y,x)
        npy_size (int): Size of the sub-volume to extract
        extract_stride (int): Stride length of the stepping sample
        sample_num (int): Number to use as suffix in the npy file names
        part_name (str): Name of the subdirectory to save the npy files

    Returns:
        None
    """
    sample_start_z, sample_start_y, sample_start_x = start_coords
    for i in range(3):
        n_samples[i] = int(n_samples[i] * npy_size / extract_stride) - 1
    os.makedirs(os.path.join(sample_dir, part_name), exist_ok=True)
    for i_z in range(n_samples[0]):
        idx_z = sample_start_z + extract_stride * i_z
        for i_y in range(n_samples[1]):
            idx_y = sample_start_y + extract_stride * i_y
            for i_x in range(n_samples[2]):
                idx_x = sample_start_x + extract_stride * i_x
                sample = data[idx_z:idx_z + npy_size, idx_y:idx_y + npy_size,
                              idx_x:idx_x + npy_size]
                if part_name == 'map_sample':
                    np.save(
                        os.path.join(sample_dir, part_name,
                                     f"map.{sample_num}.npy"), sample)
                else:
                    np.save(
                        os.path.join(sample_dir, part_name,
                                     f"model_{part_name}.{sample_num}.npy"),
                        sample)

                sample_num += 1


In [11]:
sample_num = 0 
# Create npy files from map_data
split_to_npy(map_data, sample_dir, start_coords, n_samples, npy_size,
                sample_num, 'map_sample')

In [12]:
# Sample the selected parts of the model to create the numpy array, record the number of each tag in each part
num_of_tag_in_each_part = {}  # Save number of each tag in each part


In [13]:
def atom_coord_cif(structure, RESIDUE=None, ATOM=None):
    """
    Returns the atomic coordinates from a PDB structure for specific residues and atoms.

    Args:
        structure (Structure): PDB structure.
        RESIDUE (list, optional): List of residue names to select. Defaults to None (all residues).
        ATOM (list, optional): List of atom names to select. Defaults to None (all atoms).

    Returns:
        list: List of atomic coordinates as tuples (z, y, x).
    """
    coords = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if RESIDUE is not None and residue.name not in RESIDUE:
                    continue
                for atom in residue:
                    if ATOM is not None and atom.name not in ATOM:
                        continue
                    # coords.append(
                    #     (int(round(atom.pos.z)), int(round(atom.pos.y)),
                    #      int(round(atom.pos.x)))
                    # )  
                    # coords.append(atom.pos), 
                    coords.append([atom.pos.z, atom.pos.y, atom.pos.x])
    return coords

In [16]:
def tag_npy(model_data,
            part_coords,
            tag_id,
            dis_array=np.array([]),
            atom_grid_radius=1.5):
    """
    Add tags to 3D grid points surrounding given part coordinates.

    Args:
        model_data (ndarray): A 3D numpy array representing the 3D grid.
        part_coords (list of tuples): A list of tuples representing the part coordinates (z, y, x).
        tag_id (int): The tag value to apply to the tagged points.
        atom_grid_radius (int, optional): The radius of the grid to tag around the part coordinates. Defaults to 2.

    Returns:
        model_data (ndarray): The modified model data with the tagged points.
        dis_array (ndarray): The modified model data with the tagged points.
    """

    dis_array = np.full(model_data.shape, atom_grid_radius**2)

    index_grid = int(atom_grid_radius) + 1
    extension = range(-index_grid, index_grid + 1)
    arounds = []
    for dz in extension:
        for dy in extension:
            for dx in extension:
                dist = dz**2 + dy**2 + dx**2
                if dist <= atom_grid_radius**2:
                    arounds.append([dz, dy, dx])
    arounds = np.array(arounds)
    arounds = np.vstack((arounds, arounds + np.array([0, 0, 1])))
    arounds = np.vstack((arounds, arounds + np.array([0, 1, 0])))
    arounds = np.vstack((arounds, arounds + np.array([1, 0, 0])))
    arounds = np.unique(arounds, axis=0).astype(int)

    for coord in np.array(part_coords):
        floor_coord = np.floor(coord).astype(int)
        for around in arounds:
            around_coord = floor_coord + around
            dist = np.sum((around_coord - coord) ** 2)
            if dist <= dis_array[around_coord[0], around_coord[1], around_coord[2]]:
                model_data[around_coord[0], around_coord[1], around_coord[2]] = tag_id
                dis_array[around_coord[0], around_coord[1], around_coord[2]] = dist

    return model_data, dis_array


In [17]:
# part_coords = None
model_data = np.zeros(cube_size, np.int8)
num_of_tag = []

part_name = "key_atoms"

residues_protein = [
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLU", "GLN", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL"
]
residues_rna = ['A', 'G', 'C', 'U']

atoms_sugar_ring_new = ["C4'", "O4'", "C3'", "C2'", "C1'"]
# key_atoms = [["CA"], ["C"], ["N"], ['P'], atoms_sugar_ring_new]
key_atoms = [["CA"], ['P'], atoms_sugar_ring_new]
# Tag CA, P, sugar atoms with 1, 2, 3 repectively
dis_array = np.array([])
for base_tag in range(1, len(key_atoms)+1):
    if key_atoms[base_tag - 1] in [["CA"],]:
        part_coords = atom_coord_cif(structure, residues_protein,
                                    key_atoms[base_tag - 1])
    else:
        part_coords = atom_coord_cif(structure, residues_rna,
                                     key_atoms[base_tag - 1])
    model_data, dis_array = tag_npy(model_data, part_coords, base_tag, dis_array)

In [35]:
def protein_2nd_structure_lists(structure):
    helices, sheets, = [], []
    # helices = [[hlx_chain1, start.res_id.seqid.num, end.res_id.seqid.num],
    #            [hlx_chain2, start.res_id.seqid.num, end.res_id.seqid.num],
    #            ... ]

    for helix_index in range(len(structure.helices)):
        start = structure.helices[helix_index].start.res_id.seqid.num
        end = structure.helices[helix_index].end.res_id.seqid.num
        chain_name = structure.helices[helix_index].start.chain_name
        helices.append([chain_name, start, end])

    for sheet_index in range(len(structure.sheets)):
        for strand_index in range(len(structure.sheets[sheet_index].strands)):
            start = structure.sheets[sheet_index].strands[strand_index].start.res_id.seqid.num
            end = structure.sheets[sheet_index].strands[strand_index].end.res_id.seqid.num
            chain_name = structure.sheets[sheet_index].strands[strand_index].start.chain_name
            sheets.append([chain_name, start, end])

    return helices, sheets

In [38]:
def atom_coord_cif_protein_secondary(structure, list, ATOM=None):
    coords = []
    chain_name = [row[0] for row in list]
    for model in structure:
        for chain in model:
            if chain_name is not None and chain.name not in chain_name:
                continue
            for index in range(len(chain_name)):
                if chain.name != chain_name[index]:
                    continue
                chain_info = list[index]
                for residue in chain:
                    if residue.seqid.num in range(chain_info[1], chain_info[2]+1):
                        for atom in residue:
                            if ATOM is not None and atom.name not in ATOM:
                                continue
                            coords.append([atom.pos.z, atom.pos.y, atom.pos.x])
    return coords


In [39]:
# part_coords = None
model_data = np.zeros(cube_size, np.int8)
num_of_tag = []

part_name = "secondary_strctures"

residues_rna = ['A', 'G', 'C', 'U']
atoms_protein_backbone = ['CA', 'N', 'C']
atoms_rna_backbone = [
    "P", "OP1", "OP2", "O5'", "C5'", "C4'", "O4'", "C3'", "O3'", "C2'", "O2'",
    "C1'"
]

helices, sheets = protein_2nd_structure_lists(structure)

# Tag key atoms(CA+C+N) in helices with 1, in sheets with 2
# Tag key atoms(P+sugar) in rna with 3
dis_array = np.array([])

# for helices and sheets
part_coords = atom_coord_cif_protein_secondary(structure, helices, atoms_protein_backbone)
model_data, dis_array = tag_npy(model_data, part_coords, 1, dis_array)
part_coords = atom_coord_cif_protein_secondary(structure, sheets, atoms_protein_backbone)
model_data, dis_array = tag_npy(model_data, part_coords, 2, dis_array)
# for rna
part_coords = atom_coord_cif(structure, residues_rna, atoms_rna_backbone)
model_data, dis_array = tag_npy(model_data, part_coords, 3, dis_array)
   



In [40]:
GENERATET_MRC_TEST = True
CLASSES = 3
if GENERATET_MRC_TEST is True:
    # Generate TEST.mrc for every part
    for base_tag in range(1, CLASSES + 1):
        out_map = f"{map_path.split('.mrc')[0]}_EXAMPLE_{part_name}_{base_tag}.mrc"
        print("=> Writing new map")
        shutil.copy(map_path, out_map)
        with mrcfile.open(out_map, mode='r+') as mrc:
            TEST_data = np.zeros_like(model_data)
            TEST_data = np.where(model_data == base_tag, model_data, 0)
            mrc.set_data(TEST_data)
            # mrc.header.mz = model_data.shape[0]
        print("New map is writen.")


=> Writing new map
New map is writen.
=> Writing new map
New map is writen.
=> Writing new map
New map is writen.
